# Build enriched dataset
This notebook does two things:
1. Computes **genre popularity** per market per year from the Charts + Artists data
2. Builds the **enriched model dataset** by joining genre networks + distances + genre popularity

## Config & imports

In [1]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub/Bachelor


In [2]:
import sys
sys.path.append('embeddings')

import os
import ast
import glob
import pickle
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from lorentz import Lorentz

MGD_ROOT = 'datasets'  

CHARTS_DIR = os.path.join(MGD_ROOT, 'Charts')
ARTISTS_CSV = os.path.join(MGD_ROOT, 'Artists', 'spotify_artists_info_complete.csv')
GENRE_NET_DIR = os.path.join(MGD_ROOT, 'Original')


CKPT = 'ckpt/final_enao_graph.ckpt'
VOCAB = 'enao_vocab.pkl' 
DIM = 2

MARKETS = ['au', 'br', 'ca', 'de', 'fr', 'gb', 'global', 'jp', 'us']
YEARS = [2017, 2018, 2019]

# output
OUTPUT_POPULARITY = 'prediction_model/genre_popularity.csv'
OUTPUT_RANKING = 'prediction_model/genre_ranking.csv'
OUTPUT_DATASET = 'prediction_model/model_dataset.csv'

/Users/mikolajandrzejewski/anaconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (6.0.0.post1)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


enao_graph


## 1. Load embeddings

In [3]:
genres_list = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres_list)
genre_to_idx = {g: i for i, g in enumerate(genres_list)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

lorentz_table = net.table.weight.data.cpu().numpy()

def lorentz_distance(u, v):
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    return lorentz_table[genre_to_idx[genre_name] + 1]

print(f'Loaded {n_items} genre embeddings')

Loaded 6280 genre embeddings


## 2. Build genre popularity per market per year

For each (market, year) we sum the weekly streams of every chart song, attributed to each genre of that song's artist.

```
genre_popularity(genre, market, year) =
    sum of streams across all weeks
    for all songs whose artist has that genre
```

In [4]:
artists_df = pd.read_csv(ARTISTS_CSV, sep='\t')

# parse genres from string representation of list
def parse_genres(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

artists_df['genres_parsed'] = artists_df['genres'].apply(parse_genres)

# build name -> genres lookup
artist_genres = dict(zip(artists_df['name'].str.strip(),
                         artists_df['genres_parsed']))

print(f'Artists loaded: {len(artist_genres)}')
print('Sample:', list(artist_genres.items())[:3])

Artists loaded: 3584
Sample: [('AURORA', ['norwegian pop']), ('MHD', ['francoton', 'french hip hop', 'pop urbaine']), ('Chuck Berry', ['blues rock', 'classic rock', 'folk rock', 'rock', 'rock-and-roll', 'rockabilly', 'roots rock', 'soul'])]


In [5]:
from collections import defaultdict

popularity_rows = []
skipped_artists = set()

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        weekly_files = glob.glob(os.path.join(charts_path, '*.csv'))
        if not weekly_files:
            continue

        genre_streams = defaultdict(float)

        for fpath in weekly_files:
            try:
                chart = pd.read_csv(fpath, sep='\t')
            except Exception:
                continue

            for _, row in chart.iterrows():
                artist_name = str(row['artist']).strip()
                streams = row['streams']
                genres = artist_genres.get(artist_name, [])

                if not genres:
                    skipped_artists.add(artist_name)
                    continue

                for genre in genres:
                    genre_streams[genre] += streams

        for genre, total_streams in genre_streams.items():
            popularity_rows.append({
                'market':       market,
                'year':         year,
                'genre':        genre,
                'total_streams': total_streams
            })

    print(f'{market} done')

popularity_df = pd.DataFrame(popularity_rows)
popularity_df.to_csv(OUTPUT_POPULARITY, index=False)

print(f'\nGenre popularity table: {len(popularity_df)} rows')
print(f'Artists with no genres (skipped): {len(skipped_artists)}')
print()
print(popularity_df[popularity_df['market'] == 'us'][popularity_df['year'] == 2019]
      .sort_values('total_streams', ascending=False).head(10).to_string(index=False))

au done
br done
ca done
de done
fr done
gb done
global done
jp done
us done

Genre popularity table: 6258 rows
Artists with no genres (skipped): 131

market  year            genre  total_streams
    us  2019              pop   1.201314e+10
    us  2019              rap   1.070258e+10
    us  2019          pop rap   7.501231e+09
    us  2019      melodic rap   7.103647e+09
    us  2019             trap   5.737092e+09
    us  2019    post-teen pop   5.307230e+09
    us  2019        dance pop   4.593115e+09
    us  2019          hip hop   3.516477e+09
    us  2019       electropop   2.736183e+09
    us  2019 southern hip hop   2.354967e+09


/var/folders/4x/l0sbrhlx0kx4f0hwzx7x2yqr0000gn/T/ipykernel_27606/2598149899.py:53: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(popularity_df[popularity_df['market'] == 'us'][popularity_df['year'] == 2019]


## 2b. Compute ranking-based popularity

In addition to stream-based popularity, we compute ranking-based metrics:
- `avg_ranking`: Average chart position (lower = better)
- `ranking_score`: Weighted score (rank 1 = 200 pts, rank 200 = 1 pt; higher = better)
- `n_chart_appearances`: Number of weeks on charts

In [8]:
ranking_rows = []

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        # Collect rankings for each genre
        genre_rankings = defaultdict(list)
        
        week_files = sorted(glob.glob(os.path.join(charts_path, '*.csv')))
        skipped_files = 0
        for week_file in week_files:
            try:
                # Use tab separator (these are TSV files, not CSV!)
                week_df = pd.read_csv(week_file, sep='\t')
                
                # Column names are lowercase: 'position' and 'artist'
                if 'position' not in week_df.columns or 'artist' not in week_df.columns:
                    skipped_files += 1
                    continue
                    
            except Exception as e:
                skipped_files += 1
                continue
            
            for _, row in week_df.iterrows():
                try:
                    artist_name = str(row['artist']).strip()
                    rank = int(row['position'])
                except:
                    continue
                
                if artist_name not in artist_genres:
                    continue
                    
                for genre in artist_genres[artist_name]:
                    genre_rankings[genre].append(rank)
        
        if skipped_files > 0:
            print(f'  {market} {year}: skipped {skipped_files} files')
        
        # Compute metrics
        for genre, ranks in genre_rankings.items():
            avg_rank = np.mean(ranks)
            weighted_score = sum(200 - r + 1 for r in ranks)
            
            ranking_rows.append({
                'market': market,
                'year': year,
                'genre': genre,
                'avg_ranking': avg_rank,
                'ranking_score': weighted_score,
                'n_chart_appearances': len(ranks)
            })
    
    print(f'{market} done')

ranking_df = pd.DataFrame(ranking_rows)

print(f'\nGenre ranking table: {len(ranking_df)} rows')
if len(ranking_df) > 0:
    print('Sample (US 2019):')  
    sample = ranking_df[(ranking_df['market'] == 'us') & (ranking_df['year'] == 2019)]
    if len(sample) > 0:
        print(sample.sort_values('ranking_score', ascending=False).head(10).to_string(index=False))

ranking_df.to_csv(OUTPUT_RANKING, index=False)
print(f'\nSaved to: {OUTPUT_RANKING}')

au done
br done
ca done
de done
fr done
gb done
global done
jp done
us done

Genre ranking table: 6258 rows
Sample (US 2019):
market  year            genre  avg_ranking  ranking_score  n_chart_appearances
    us  2019              pop    98.700941         423825                 4143
    us  2019              rap    98.904828         365807                 3583
    us  2019          pop rap    95.746993         262501                 2494
    us  2019      melodic rap    91.394962         243652                 2223
    us  2019             trap   103.207352         207516                 2122
    us  2019    post-teen pop    97.732173         183920                 1781
    us  2019        dance pop    96.674000         156489                 1500
    us  2019          hip hop   107.925581         120066                 1290
    us  2019       electropop    98.963806          98669                  967
    us  2019 southern hip hop    99.834306          86699                  857

Save

## 3. Build the enriched model dataset

For each (market, year) genre network file we:
- Remove self-loops
- Compute hyperbolic distance for each pair
- Attach `popularity_source` and `popularity_target` from the popularity table
- Add `market` and `year` columns

Then stack all markets and years into one CSV.

In [9]:
pop_lookup = {}
for _, row in popularity_df.iterrows():
    pop_lookup[(row['market'], row['year'], row['genre'])] = row['total_streams']

# NEW: Ranking lookups
rank_lookup = {}
rank_score_lookup = {}
for _, row in ranking_df.iterrows():
    rank_lookup[(row['market'], row['year'], row['genre'])] = row['avg_ranking']
    rank_score_lookup[(row['market'], row['year'], row['genre'])] = row['ranking_score']

all_rows = []
skipped_no_embedding = 0
skipped_no_popularity = 0

for market in MARKETS:
    for year in YEARS:
        net_path = os.path.join(GENRE_NET_DIR, market,
                                f'{market}-genre_network-{year}.csv')
        if not os.path.exists(net_path):
            print(f'  Missing genre network: {net_path}')
            continue

        gn = pd.read_csv(net_path, sep='\t')
        gn = gn[gn['source'] != gn['target']]  # remove self-loops

        for _, row in gn.iterrows():
            src, tgt = row['source'], row['target']

            # hyperbolic distance
            u = get_embedding(src)
            v = get_embedding(tgt)
            if u is None or v is None:
                skipped_no_embedding += 1
                continue

            dist = lorentz_distance(u, v)

            # genre popularity (streams)
            pop_src = pop_lookup.get((market, year, src))
            pop_tgt = pop_lookup.get((market, year, tgt))

            if pop_src is None or pop_tgt is None:
                skipped_no_popularity += 1
                continue

            # NEW: Genre ranking
            rank_src = rank_lookup.get((market, year, src))
            rank_tgt = rank_lookup.get((market, year, tgt))
            rank_score_src = rank_score_lookup.get((market, year, src))
            rank_score_tgt = rank_score_lookup.get((market, year, tgt))

            all_rows.append({
                'market':       market,
                'year':         year,
                'source':       src,
                'target':       tgt,
                'weight':       row['weight'],
                'avg_streams':  row['avg_streams'],
                'distance':     dist,
                
                # Streams-based popularity
                'popularity_source':   pop_src,
                'popularity_target':   pop_tgt,
                
                # NEW: Ranking-based popularity
                'ranking_source':      rank_src,
                'ranking_target':      rank_tgt,
                'ranking_score_source': rank_score_src,
                'ranking_score_target': rank_score_tgt,
                
                # Log transforms (streams)
                'log_streams':  np.log10(row['avg_streams'] + 1),
                'log_popularity_source':  np.log10(pop_src + 1),
                'log_popularity_target':  np.log10(pop_tgt + 1),
                'log_weight':   np.log10(row['weight'] + 1),
                
                # NEW: Log transforms (ranking)
                'log_ranking_score_source': np.log10(rank_score_src + 1) if rank_score_src else None,
                'log_ranking_score_target': np.log10(rank_score_tgt + 1) if rank_score_tgt else None,
            })

model_df = pd.DataFrame(all_rows)

# Add normalized distances
print('\nNormalizing distances...')

# Per-market normalization
model_df['distance_norm'] = 0.0
for market in model_df['market'].unique():
    mask = model_df['market'] == market
    scaler = MinMaxScaler()
    model_df.loc[mask, 'distance_norm'] = scaler.fit_transform(
        model_df.loc[mask, ['distance']]
    ).flatten()

# Global normalization
scaler_global = MinMaxScaler()
model_df['distance_norm_global'] = scaler_global.fit_transform(
    model_df[['distance']]
).flatten()

print(f'Added normalized distances (per-market and global)')
print(f"  Per-market range: {model_df['distance_norm'].min():.4f} - {model_df['distance_norm'].max():.4f}")
print(f"  Global range: {model_df['distance_norm_global'].min():.4f} - {model_df['distance_norm_global'].max():.4f}")
print()

model_df.to_csv(OUTPUT_DATASET, index=False)

print(f'Model dataset: {len(model_df)} rows')
print(f'Skipped (no embedding): {skipped_no_embedding}')
print(f'Skipped (no popularity): {skipped_no_popularity}')
print()
print('Rows per market:')
print(model_df['market'].value_counts().to_string())
print()
print('Rows per year:')
print(model_df['year'].value_counts().sort_index().to_string())


Normalizing distances...
Added normalized distances (per-market and global)
  Per-market range: 0.0000 - 1.0000
  Global range: 0.0000 - 1.0000

Model dataset: 43661 rows
Skipped (no embedding): 4265
Skipped (no popularity): 13067

Rows per market:
market
ca        6304
global    6022
gb        5995
us        5765
au        5209
de        5056
fr        3665
br        3045
jp        2600

Rows per year:
year
2017    15316
2018    15461
2019    12884


No popularity (13,067 rows skipped). These genres DO exist in the Artists file, but the artists who play them never appeared in the top 200 charts for that specific market/year. For example album rock artists like John Lennon and Journey appear in the US charts, but not in Germany or Brazil. So album rock has zero streams in those markets — it genuinely has no popularity there.

## 4. Quick sanity check

In [10]:
print('=== Dataset summary ===')
print(model_df[['avg_streams', 'distance', 'popularity_source', 'popularity_target', 
                'ranking_score_source', 'ranking_score_target', 'weight']]
      .describe().round(2))
print()
print('=== Sample rows ===')
print(model_df[['market', 'year', 'source', 'target',
                'distance', 'log_popularity_source', 'log_popularity_target',
                'log_ranking_score_source', 'log_ranking_score_target',
                'log_weight', 'log_streams']]
      .head(10).to_string(index=False))
print()
print('=== Missing values ===')
print(model_df.isnull().sum())
print()
print('=== Ranking vs Streams correlation ===')
# Check correlation between ranking and streams
df_complete = model_df.dropna(subset=['ranking_score_source', 'ranking_score_target'])
if len(df_complete) > 0:
    corr_src = df_complete[['ranking_score_source', 'popularity_source']].corr().iloc[0,1]
    corr_tgt = df_complete[['ranking_score_target', 'popularity_target']].corr().iloc[0,1]
    print(f"Correlation (ranking_score_source, popularity_source): {corr_src:.4f}")
    print(f"Correlation (ranking_score_target, popularity_target): {corr_tgt:.4f}")
    print(f"Rows with ranking data: {len(df_complete)} / {len(model_df)} ({100*len(df_complete)/len(model_df):.1f}%)")

=== Dataset summary ===
        avg_streams  distance  popularity_source  popularity_target  \
count  4.366100e+04  43661.00       4.366100e+04       4.366100e+04   
mean   1.661034e+07      4.20       1.002963e+09       1.539252e+09   
std    4.223581e+07      2.12       3.403722e+09       4.452900e+09   
min    1.535100e+04      0.02       1.535100e+04       1.535100e+04   
25%    1.268182e+06      2.61       1.579184e+07       2.588379e+07   
50%    4.028020e+06      4.18       9.055472e+07       1.630749e+08   
75%    1.164331e+07      5.99       5.083869e+08       8.842194e+08   
max    8.583335e+08      9.20       4.357425e+10       4.357425e+10   

       ranking_score_source  ranking_score_target    weight  
count              43661.00              43661.00  43661.00  
mean               54859.29              79446.63      6.41  
std               101379.41             124170.04     23.40  
min                    1.00                  1.00      1.00  
25%                 2312.0